# Portfolio Backtesting UI

        This is the simple user-facing version. Run the setup cell once, then use the visual dashboard to test investment strategies without editing the engine code.


## Step 1 - Run Setup


In [ ]:

import subprocess
import sys
import warnings
from dataclasses import dataclass


def install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


install(["pandas>=2.2.0", "numpy>=1.26.0", "yfinance>=0.2.40", "plotly>=5.20.0", "ipywidgets>=8.1.0"])

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from IPython.display import clear_output, display
import ipywidgets as widgets

warnings.filterwarnings("ignore")


@dataclass(frozen=True)
class UserConfig:
    tickers: tuple[str, ...]
    benchmark: str
    start: str
    end: str | None
    initial_cash: float
    strategy: str
    rebalance_frequency: str
    lookback: int
    top_n: int
    commission_bps: float
    slippage_bps: float
    risk_free_rate: float


def clean_tickers(value):
    tickers = tuple(item.strip().upper() for item in value.replace("\n", ",").split(",") if item.strip())
    if not tickers:
        raise ValueError("Enter at least one ticker.")
    return tickers


def download_prices(config):
    symbols = list(dict.fromkeys([*config.tickers, config.benchmark]))
    data = yf.download(
        symbols,
        start=config.start,
        end=config.end or None,
        auto_adjust=True,
        group_by="ticker",
        progress=False,
        threads=True,
        repair=True,
    )
    if data.empty:
        raise RuntimeError("Yahoo Finance returned no data. Try different tickers or dates.")

    if isinstance(data.columns, pd.MultiIndex):
        level_0 = data.columns.get_level_values(0)
        close = data.xs("Close", axis=1, level=0) if "Close" in level_0 else data.xs("Close", axis=1, level=1)
    else:
        close = data[["Close"]].rename(columns={"Close": symbols[0]})

    close.columns = [str(col).upper() for col in close.columns]
    close = close.sort_index().replace([np.inf, -np.inf], np.nan).ffill().dropna(how="all")
    missing = sorted(set(config.tickers).difference(close.columns))
    if missing:
        raise RuntimeError(f"Missing usable prices for: {missing}")
    return close


def rebalance_mask(index, frequency):
    periods = pd.Series(pd.DatetimeIndex(index).to_period(frequency), index=index)
    mask = periods.ne(periods.shift(-1)).fillna(True)
    mask.iloc[0] = True
    return mask


def normalize_weights(scores):
    scores = scores.clip(lower=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    totals = scores.sum(axis=1).replace(0, np.nan)
    return scores.div(totals, axis=0).fillna(0.0)


def build_weights(prices, returns, config):
    universe = list(config.tickers)
    mask = rebalance_mask(prices.index, config.rebalance_frequency)

    if config.strategy == "Equal Weight":
        available = prices[universe].notna().astype(float)
        weights = available.div(available.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

    elif config.strategy == "Dual Momentum":
        momentum = prices[universe].pct_change(config.lookback, fill_method=None)
        selected = momentum.where(momentum > 0, 0.0)
        ranks = selected.rank(axis=1, ascending=False, method="first")
        weights = selected.where(ranks <= config.top_n, 0.0).gt(0.0).astype(float)
        weights = normalize_weights(weights)

    elif config.strategy == "Volatility-Target Momentum":
        momentum = prices[universe].pct_change(config.lookback, fill_method=None).clip(lower=0)
        volatility = returns[universe].rolling(63, min_periods=21).std() * np.sqrt(252)
        trend = prices[universe] > prices[universe].rolling(200, min_periods=60).mean()
        weights = normalize_weights((momentum / volatility.replace(0, np.nan)).where(trend, 0.0))

    else:
        raise ValueError(f"Unknown strategy: {config.strategy}")

    return weights.where(mask, np.nan).ffill().fillna(0.0)


def run_backtest(prices, config):
    returns = prices.pct_change(fill_method=None).fillna(0.0)
    target_weights = build_weights(prices, returns, config)
    execution_weights = target_weights.shift(1).fillna(0.0)
    cost_rate = (config.commission_bps + config.slippage_bps) / 10_000

    portfolio_returns = (execution_weights[list(config.tickers)] * returns[list(config.tickers)]).sum(axis=1)
    turnover = execution_weights.diff().abs().sum(axis=1).fillna(0.0)
    cost_returns = turnover * cost_rate
    net_returns = portfolio_returns - cost_returns
    equity = config.initial_cash * (1 + net_returns).cumprod()

    benchmark_returns = returns[config.benchmark].reindex(net_returns.index).fillna(0.0) if config.benchmark in returns else None
    benchmark_equity = config.initial_cash * (1 + benchmark_returns).cumprod() if benchmark_returns is not None else None

    return {
        "prices": prices,
        "returns": returns,
        "weights": execution_weights,
        "portfolio_returns": net_returns,
        "equity": equity,
        "benchmark_returns": benchmark_returns,
        "benchmark_equity": benchmark_equity,
        "turnover": turnover,
    }


def drawdown(equity):
    return equity / equity.cummax() - 1


def metrics(result, config):
    r = result["portfolio_returns"].fillna(0.0)
    periods = 252
    equity = result["equity"]
    dd = drawdown(equity)
    years = max(len(r) / periods, 1 / periods)
    cagr = (equity.iloc[-1] / config.initial_cash) ** (1 / years) - 1
    vol = r.std(ddof=0) * np.sqrt(periods)
    rf_daily = (1 + config.risk_free_rate) ** (1 / periods) - 1
    sharpe = ((r - rf_daily).mean() * periods) / ((r - rf_daily).std(ddof=0) * np.sqrt(periods)) if r.std(ddof=0) > 0 else np.nan
    sortino_den = r.where(r < rf_daily, 0).std(ddof=0) * np.sqrt(periods)
    sortino = ((r - rf_daily).mean() * periods) / sortino_den if sortino_den > 0 else np.nan
    calmar = cagr / abs(dd.min()) if dd.min() < 0 else np.nan
    return pd.DataFrame(
        {
            "Metric": ["Final Value", "Total Return", "CAGR", "Sharpe", "Sortino", "Calmar", "Max Drawdown", "Annual Volatility", "Hit Rate"],
            "Value": [
                f"${equity.iloc[-1]:,.0f}",
                f"{equity.iloc[-1] / config.initial_cash - 1:.2%}",
                f"{cagr:.2%}",
                f"{sharpe:.2f}",
                f"{sortino:.2f}",
                f"{calmar:.2f}",
                f"{dd.min():.2%}",
                f"{vol:.2%}",
                f"{(r > 0).mean():.2%}",
            ],
        }
    )


def chart_equity(result, config):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=result["equity"].index, y=result["equity"], mode="lines", name=config.strategy, line=dict(width=3, color="#0F6B5B")))
    if result["benchmark_equity"] is not None:
        fig.add_trace(go.Scatter(x=result["benchmark_equity"].index, y=result["benchmark_equity"], mode="lines", name=config.benchmark, line=dict(width=2, color="#4C566A", dash="dot")))
    fig.update_layout(title="Equity Curve", template="plotly_white", yaxis_title="Portfolio Value", hovermode="x unified", height=430)
    return fig


def chart_drawdown(result):
    dd = drawdown(result["equity"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dd.index, y=dd, mode="lines", fill="tozeroy", name="Drawdown", line=dict(width=2, color="#B23A48")))
    fig.update_layout(title="Drawdown", template="plotly_white", yaxis_tickformat=".0%", hovermode="x unified", height=330)
    return fig


def chart_weights(result, config):
    fig = go.Figure()
    for ticker in config.tickers:
        fig.add_trace(go.Scatter(x=result["weights"].index, y=result["weights"][ticker], mode="lines", stackgroup="one", name=ticker))
    fig.update_layout(title="Portfolio Weights", template="plotly_white", yaxis_tickformat=".0%", hovermode="x unified", height=360)
    return fig


def chart_monthly_returns(result):
    monthly = (1 + result["portfolio_returns"]).resample("ME").prod() - 1
    frame = monthly.to_frame("return")
    frame["year"] = frame.index.year
    frame["month"] = frame.index.strftime("%b")
    order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    pivot = frame.pivot(index="year", columns="month", values="return").reindex(columns=order)
    fig = go.Figure(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns,
            y=pivot.index,
            colorscale="RdYlGn",
            zmid=0,
            hovertemplate="Year=%{y}<br>Month=%{x}<br>Return=%{z:.2%}<extra></extra>",
        )
    )
    fig.update_layout(title="Monthly Returns", template="plotly_white", height=330)
    return fig


print("Setup complete. Run the dashboard cell below.")


## Step 2 - Use The Dashboard


In [ ]:

tickers_box = widgets.Text(
    value="SPY, QQQ, TLT, GLD",
    description="Tickers",
    layout=widgets.Layout(width="520px"),
)
benchmark_box = widgets.Text(value="SPY", description="Benchmark", layout=widgets.Layout(width="260px"))
start_box = widgets.Text(value="2015-01-01", description="Start", layout=widgets.Layout(width="260px"))
end_box = widgets.Text(value="", description="End", placeholder="blank = today", layout=widgets.Layout(width="260px"))
cash_box = widgets.FloatText(value=100000, description="Cash", layout=widgets.Layout(width="260px"))
strategy_dropdown = widgets.Dropdown(
    options=["Volatility-Target Momentum", "Dual Momentum", "Equal Weight"],
    value="Volatility-Target Momentum",
    description="Strategy",
    layout=widgets.Layout(width="360px"),
)
frequency_dropdown = widgets.Dropdown(
    options=[("Monthly", "M"), ("Weekly", "W"), ("Quarterly", "Q")],
    value="M",
    description="Rebalance",
    layout=widgets.Layout(width="260px"),
)
lookback_slider = widgets.IntSlider(value=126, min=21, max=252, step=21, description="Lookback", layout=widgets.Layout(width="420px"))
top_n_slider = widgets.IntSlider(value=2, min=1, max=6, step=1, description="Top N", layout=widgets.Layout(width="420px"))
commission_box = widgets.FloatText(value=1.0, description="Comm bps", layout=widgets.Layout(width="260px"))
slippage_box = widgets.FloatText(value=2.0, description="Slip bps", layout=widgets.Layout(width="260px"))
rf_box = widgets.FloatText(value=0.02, description="Risk-free", layout=widgets.Layout(width="260px"))
run_button = widgets.Button(description="Run Backtest", button_style="success", icon="play", layout=widgets.Layout(width="180px"))
output = widgets.Output()

controls = widgets.VBox(
    [
        widgets.HTML("<h2>Portfolio Strategy Tester</h2>"),
        widgets.HTML("<b>1. Choose assets and settings. 2. Click Run Backtest. 3. Read the charts.</b>"),
        widgets.HBox([tickers_box, benchmark_box]),
        widgets.HBox([start_box, end_box, cash_box]),
        widgets.HBox([strategy_dropdown, frequency_dropdown]),
        widgets.HBox([lookback_slider, top_n_slider]),
        widgets.HBox([commission_box, slippage_box, rf_box]),
        run_button,
    ]
)


def on_run_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            config = UserConfig(
                tickers=clean_tickers(tickers_box.value),
                benchmark=benchmark_box.value.strip().upper(),
                start=start_box.value.strip(),
                end=end_box.value.strip() or None,
                initial_cash=float(cash_box.value),
                strategy=strategy_dropdown.value,
                rebalance_frequency=frequency_dropdown.value,
                lookback=int(lookback_slider.value),
                top_n=int(top_n_slider.value),
                commission_bps=float(commission_box.value),
                slippage_bps=float(slippage_box.value),
                risk_free_rate=float(rf_box.value),
            )
            if config.top_n > len(config.tickers):
                raise ValueError("Top N cannot be larger than the number of tickers.")

            print("Downloading prices and running backtest...")
            prices = download_prices(config)
            result = run_backtest(prices, config)
            clear_output(wait=True)

            display(widgets.HTML(f"<h3>{config.strategy} Results</h3>"))
            display(metrics(result, config))
            chart_equity(result, config).show()
            chart_drawdown(result).show()
            chart_weights(result, config).show()
            chart_monthly_returns(result).show()

            reports = {
                "metrics": metrics(result, config),
                "equity": result["equity"],
                "weights": result["weights"],
                "returns": result["portfolio_returns"],
            }
            print("Backtest complete. The charts above are the user-facing output.")

        except Exception as exc:
            clear_output(wait=True)
            print(f"Could not run backtest: {exc}")


run_button.on_click(on_run_clicked)
display(controls, output)


## How Users Use This

        Users only change the visible inputs: tickers, benchmark, dates, cash amount, strategy, rebalance frequency, and strategy parameters. Then they click **Run Backtest** and read the charts.
